# Load Source Data - Credit Union Merger Demo

Loads the synthetic CSV data for the two legacy credit unions and the enterprise reference data into **Delta tables** in three lakehouses:

| Lakehouse | Plays the role of | Tables |
|---|---|---|
| `Contoso_CU` | Legacy credit union #1 source system | `dim_branch`, `fact_monthly_kpi`, `fact_driver_breakdown`, `glossary_kpi_definitions` |
| `Fabrikam_CU` | Legacy credit union #2 source system (different schema) | `branches`, `branch_month_metrics`, `branch_driver_scores`, `kpi_glossary` |
| `Enterprise_Ref` | Enterprise reference data | `glossary_kpi_definitions` (certified), `security_user_scope` (RLS personas), sample platform telemetry |

**Before you run:** create the three lakehouses in this workspace (see `docs/02-setup.md`), then set the parameters below.

> In production these sources would typically arrive through **Mirroring** (Snowflake, Azure SQL, Oracle and others) rather than CSV files. The lakehouses stand in for the mirrored databases so the demo installs anywhere.

In [ ]:
# ---------------- Parameters ----------------
# Where to read the CSV files from:
#   "github" - straight from the repository's raw URL (public repo, outbound internet allowed)
#   "files"  - from the Files area of a lakehouse: <FILES_LAKEHOUSE>/Files/data/<folder>/<table>.csv
DATA_SOURCE     = "github"
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/<your-org>/<your-repo>/main/data"
FILES_LAKEHOUSE = "Enterprise_Ref"

# Your Microsoft Entra tenant domain. Replaces {TENANT_DOMAIN} in the persona UPNs so row-level
# security matches the demo users you create, e.g. "contoso.onmicrosoft.com".
TENANT_DOMAIN   = "<yourtenant>.onmicrosoft.com"

# Lakehouse names (must already exist in this workspace).
LAKEHOUSES = {"contoso_cu": "Contoso_CU", "fabrikam_cu": "Fabrikam_CU", "enterprise_ref": "Enterprise_Ref"}

# Set to False if you use the optional Fabric SQL database as the Fabrikam source instead.
LOAD_FABRIKAM = True

In [ ]:
# Table manifest: folder -> table -> columns to convert to DATE / TIMESTAMP
TABLES = {
    "contoso_cu": {
        "dim_branch": {},
        "fact_monthly_kpi": {"date": ["MonthStart"]},
        "fact_driver_breakdown": {"date": ["MonthStart"]},
        "glossary_kpi_definitions": {},
    },
    "fabrikam_cu": {
        "branches": {},
        "branch_month_metrics": {"date": ["period_month"]},
        "branch_driver_scores": {"date": ["period_month"]},
        "kpi_glossary": {},
    },
    "enterprise_ref": {
        "glossary_kpi_definitions": {},
        "security_user_scope": {},
        "ops_capacity_metrics_5min": {"timestamp": ["Timestamp"]},
        "ops_refresh_history": {"timestamp": ["RefreshTimestamp"]},
        "admin_inventory_artifacts": {"date": ["LastViewedDate"]},
    },
}
# Columns that must stay text even if they look numeric
KEEP_AS_TEXT = {"BranchId", "branch_code", "ArtifactId", "State", "state_code", "region_code"}

In [ ]:
import pandas as pd

ws_id = notebookutils.runtime.context["currentWorkspaceId"]

def _prop(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

def tables_root(lakehouse_name):
    """OneLake path to the Tables folder of a lakehouse in this workspace (handles schema-enabled lakehouses)."""
    try:
        lh = notebookutils.lakehouse.get(lakehouse_name)
    except Exception as e:
        raise RuntimeError(f"Lakehouse '{lakehouse_name}' was not found in this workspace. "
                           f"Create it first (see docs/02-setup.md). Details: {e}")
    lh_id = _prop(lh, "id")
    props = _prop(lh, "properties", {}) or {}
    has_schemas = bool(_prop(props, "defaultSchema"))
    root = f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{lh_id}/Tables"
    return (root + "/dbo") if has_schemas else root

def read_source(folder, table):
    if DATA_SOURCE == "github":
        return pd.read_csv(f"{GITHUB_RAW_BASE}/{folder}/{table}.csv", dtype=str, keep_default_na=False)
    files_lh = notebookutils.lakehouse.get(FILES_LAKEHOUSE)
    path = (f"abfss://{ws_id}@onelake.dfs.fabric.microsoft.com/{_prop(files_lh, 'id')}"
            f"/Files/data/{folder}/{table}.csv")
    return spark.read.option("header", True).csv(path).toPandas().fillna("")

def type_columns(pdf, spec):
    for col in pdf.columns:
        if col in KEEP_AS_TEXT or col in spec.get("date", []) or col in spec.get("timestamp", []):
            continue
        converted = pd.to_numeric(pdf[col], errors="coerce")
        if converted.notna().all():
            pdf[col] = converted
    for col in spec.get("date", []):
        pdf[col] = pd.to_datetime(pdf[col]).dt.date
    for col in spec.get("timestamp", []):
        pdf[col] = pd.to_datetime(pdf[col])
    for col in pdf.columns:                      # plain Python strings for Spark
        if pdf[col].dtype.kind not in "iufMm" and col not in spec.get("date", []):
            pdf[col] = pdf[col].astype(object)
    return pdf

In [ ]:
results = []
for folder, tables in TABLES.items():
    if folder == "fabrikam_cu" and not LOAD_FABRIKAM:
        print("Skipping Fabrikam_CU (LOAD_FABRIKAM = False)")
        continue
    root = tables_root(LAKEHOUSES[folder])
    for table, spec in tables.items():
        pdf = read_source(folder, table)
        if table == "security_user_scope":
            pdf["UserUPN"] = pdf["UserUPN"].str.replace("{TENANT_DOMAIN}", TENANT_DOMAIN, regex=False)
        pdf = type_columns(pdf, spec)
        sdf = spark.createDataFrame(pdf)
        (sdf.write.format("delta").mode("overwrite")
            .option("overwriteSchema", "true")
            .save(f"{root}/{table}"))
        results.append((LAKEHOUSES[folder], table, sdf.count()))
        print(f"{LAKEHOUSES[folder]}.{table}: {sdf.count()} rows")

display(pd.DataFrame(results, columns=["Lakehouse", "Table", "Rows"]))

## Next steps
1. Open each lakehouse's **SQL analytics endpoint** and select **Refresh** if a table doesn't show yet (metadata sync can take a minute).
2. In the `Combined_Data` warehouse run `fabric/warehouse/01_create_objects.sql`, `02_usp_Load_Combined.sql`, then `EXEC combined.usp_Load_Combined;`
3. Check the result with `fabric/warehouse/03_validate.sql`.